# data incident responder: evaluation run

Everything runs in this notebook: the model is served by vLLM on the GPU, and the warehouse, dbt, the MCP server
and the agents run next to it. Runtime: A100 (80 GB) with high RAM.

Optional: add `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` and `LANGFUSE_BASE_URL` as Colab secrets to trace every run.

Results are written to Drive after each incident. If the runtime drops, run all again and finished incidents are skipped.

In [1]:
import os, subprocess

repo = "/content/responder"
if os.path.exists(repo):
    subprocess.run(["git", "-C", repo, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/arihaaant/data-incident-responder.git", repo], check=True)
print("code at", subprocess.run(["git", "-C", repo, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
%cd /content/responder
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
!pip uninstall -y -q torchaudio
!pip install -q -e .

code at 87a07bd runner writes the report next to each result and records the alarms

/content/responder
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.6/320.6 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.7/33.7 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.0/426.0 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import os
from google.colab import userdata

for key in ["LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_BASE_URL"]:
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        pass
print("tracing on" if os.getenv("LANGFUSE_PUBLIC_KEY") else "tracing off")

tracing on


## serve the model

In [3]:
import subprocess, time, urllib.request

model = "openai/gpt-oss-120b"
log = open("/content/vllm.log", "w")
server = subprocess.Popen(
    ["vllm", "serve", model, "--port", "8000", "--max-model-len", "32768", "--gpu-memory-utilization", "0.9",
     "--enable-auto-tool-choice", "--tool-call-parser", "openai"],
    stdout=log, stderr=subprocess.STDOUT)

t0 = time.time()
while True:
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        break
    except Exception:
        if server.poll() is not None or time.time() - t0 > 1800:
            print(open("/content/vllm.log").read()[-3000:])
            raise RuntimeError("vllm did not start")
        time.sleep(15)
print(f"model up after {(time.time() - t0) / 60:.1f} min")
os.environ["LLM_PROVIDER"], os.environ["LLM_MODEL"] = "vllm", model

model up after 7.8 min


## warehouse

In [4]:
import subprocess, sys
!python -m responder.warehouse --path kit/source.duckdb
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout[-1500:])
assert tests.returncode == 0, "tests failed, fix before running"

built kit/source.duckdb
............................................                             [100%]
44 passed in 33.24s



## runs: the responder and both baselines on the 48 development incidents

In [5]:
from google.colab import drive

drive.mount("/content/drive")
run_name = "run3"
out = f"/content/drive/MyDrive/data-incident-responder/{run_name}"
os.makedirs(out, exist_ok=True)

Mounted at /content/drive


In [6]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.run --system {system} --split dev --workers 6 --out {out}/dev-{system}


=== multi ===
[10/10/26 19:30:56] INFO     Processing request of type            ]8;id=387315;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=948787;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 19:30:56] INFO     Processing request of type            ]8;id=762711;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=77171;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py#733\733]8;;\
                             ListToolsRequest                                   
[10/10/26 19:30:56] INFO     Processing request of type            ]8;id=149082;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=902164;file:///usr/local/lib/python3.13/dist-packages/mcp/server/lowlevel/s

## results

In [7]:
for system in ["multi", "single", "rules"]:
    print(f"\n=== {system} ===")
    !python -m responder.report {out}/dev-{system}


=== multi ===
                   n    root   cause  action  resolved  tokens
overall           49   98.0%   98.0%   87.8%     87.8%   17511
duplicates        12  100.0%  100.0%   91.7%     91.7%   20790
late_data         12  100.0%  100.0%  100.0%    100.0%   21626
null_spike        11   90.9%   90.9%   90.9%     90.9%   14854
schema_drift      14  100.0%  100.0%   71.4%     71.4%   13262
dev               49   98.0%   98.0%   87.8%     87.8%   17511

resolved 43/49 (95% CI 76 to 94%), fixes passing the hidden check 29/34

=== single ===
                   n    root   cause  action  resolved  tokens
overall           50   78.0%   82.0%   88.0%     86.0%   21422
duplicates        12   91.7%   91.7%   83.3%     75.0%   25424
late_data         12   83.3%   91.7%   91.7%     91.7%   26818
null_spike        12   91.7%  100.0%   91.7%     91.7%   19058
schema_drift      14   50.0%   50.0%   85.7%     85.7%   15392
dev               50   78.0%   82.0%   88.0%     86.0%   21422

resolved 43/5